# Explainable Boosting Machines (EBM)

In [1]:
import os

print("=== TEST TMUX ===")
tmux_var = os.environ.get("TMUX")

if tmux_var:
    print("STATUS: Jupyter server is running in a TMUX session!")
    print(f"Tmux socket path: {tmux_var}")
else:
    print("STATUS: The Jupyter server is NOT inside Tmux (it runs in the global terminal).")

=== TEST TMUX ===
STATUS: Jupyter server is running in a TMUX session!
Tmux socket path: /tmp//tmux-1022/default,842946,5


## Data extraction

In [2]:
from utils.data_processing import load_and_preprocess_data

In [3]:
X, y, feature_names = load_and_preprocess_data(dataset_path="../Data/MARSIS_historical_dataset.csv", orbit_path="../Data/orbit_to_remove", keep_flux=True)

### EBM Experiment 1

In [ ]:
from interpret.glassbox import ExplainableBoostingRegressor
from utils.pipeline import pipeline

seed_exp_1 = 42

ebm_parameters = {
    "interactions": 0,
    "outer_bags": 4,
    "max_bins": 128
}

k_fold_dict_ebm, save_dir_ebm = pipeline(
    X=X, 
    y=y, 
    model_class=ExplainableBoostingRegressor, # passing Regressor class
    n_splits=10, 
    exp_name="ebm_experiment_1",
    seed_ebm=seed_exp_1,
    model_kwargs=ebm_parameters
)

=== Starting Experiment Pipeline: ebm_experiment_1 ===
Created isolated environment at: experiments/ebm_experiment_1_20260521_201953

--- Starting FOLD 0 ---
Train MSE: 7.816, Test MSE: 27.117, Train MAE: 2.135, Test MAE: 4.120

--- Starting FOLD 1 ---
Train MSE: 7.805, Test MSE: 22.069, Train MAE: 2.117, Test MAE: 3.773

--- Starting FOLD 2 ---

Execution 'ebm_experiment_1' successfully completed in 48210.48s!
Generating Learning Curves...
Note: Explainable Boosting Machine detected. Skipping Learning Curves plotting.
Experiment completely saved in: /home/emiliano/projects/project_1/Lab_XAI/Lab_XAI_pytorch/experiments/ebm_experiment_1_20260521_201953


### EBM Explanation Plots Experiment 1

#### Global explanation

In [5]:
import os
import pandas as pd
from joblib import load
from interpret import show


model_path_exp_1 = "experiments/ebm_experiment_1_20260521_201953/model_ebm_experiment_1_3.save"

# loading model
ebm_model_exp_1 = load(model_path_exp_1)
print("Loaded model successfully!")

Loaded model successfully!


In [11]:
# Inject real names into the internal attribute of the loaded model
ebm_model_exp_1.feature_names_ = feature_names

# If there are strings/categories in the dataset, EBM may want to update 
# the display names of the relationships as well. To be safe, let's update this as well:
ebm_model_exp_1.term_names_ = feature_names

# Generates the global explanation (without parameters, it will use the ones just injected)
ebm_global_exp_1 = ebm_model_exp_1.explain_global()

# Chart 1: Global Feature Importance (Summary)
print("Loading Feature Importance Graph...")
fig_summary = ebm_global_exp_1.visualize()  # without inputs it plots global summary
fig_summary.show()


Loading Feature Importance Graph...


#### Local explanation

In [12]:
# Chart 2: The curve (Shape Function) of a specific variable

# variable to be analyzed
name_feat_1 = "FM_data_altitude" 

# finding index of such variable in the list
feature_index = feature_names.index(name_feat_1)

print(f"Loading curve for feature: {name_feat_1} (Positional index: {feature_index})...")

# passes entire index to visualization function
fig_feature = ebm_global_exp_1.visualize(feature_index)
fig_feature.show()

Loading curve for feature: FM_data_altitude (Positional index: 0)...
